# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A1704/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use Logistic Regression as an interpretable starting point and Random Forest as a more flexible comparison model. The task is to rank content items by their likelihood of having the observed declining-trend label, so I evaluate ranked predictions using Precision@50. The Week 4 rule-based refresh score is the baseline. All approaches are evaluated on the same held-out clients and rows. The label indicates a historical declining trend; it does not establish that refreshing a page will improve its performance.

In [9]:
from pathlib import Path
import os

print("Current working directory:", os.getcwd())
print("\nFiles in the current directory:")
print(os.listdir("."))

print("\nSearching for the dataset...")
matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

if matches:
    for path in matches:
        print("FOUND:", path)
else:
    print("Dataset not found under /content.")

Current working directory: /content

Files in the current directory:
['.config', 'flyrank-ml-internship', 'sample_data']

Searching for the dataset...
FOUND: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [10]:
from pathlib import Path
import urllib.request

DATA_PATH = Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

url = (
    "https://raw.githubusercontent.com/"
    "A1704/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

urllib.request.urlretrieve(url, DATA_PATH)

print("Dataset downloaded:", DATA_PATH.exists())
print("File size (MB):", round(DATA_PATH.stat().st_size / (1024 * 1024), 2))

Dataset downloaded: True
File size (MB): 6.42


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 1 — Load data, validate target, and build safe features

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

RANDOM_STATE = 42
DATA_PATH = Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Check the Colab Files panel for "
        "content_refresh_anonymized.csv and update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

required = [
    "content_id", "client_id", "trend_direction",
    "impressions_90d", "days_since_last_update",
    "avg_position", "word_count"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Match the repository's documented preparation rules.
df["trend_direction"] = df["trend_direction"].fillna("unknown").astype(str)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

df = df[
    (pd.to_numeric(df["impressions_90d"], errors="coerce") > 0)
    & (pd.to_numeric(df["content_age_days"], errors="coerce") >= 90)
].drop_duplicates("content_id").reset_index(drop=True)

# Features follow the repository's model allowlist.
# Deliberately exclude trend_pct, trend_direction, the label, and identifiers.
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
log_source = {
    "log_impressions_90d": "impressions_90d",
    "log_clicks_90d": "clicks_90d",
    "log_sessions_90d": "sessions_90d",
    "log_ai_sessions_90d": "ai_sessions_90d",
}
for new_col, source_col in log_source.items():
    if source_col in df.columns:
        df[new_col] = np.log1p(
            pd.to_numeric(df[source_col], errors="coerce").clip(lower=0)
        )
        numeric_features.append(new_col)

categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier"
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

# Explicit missingness flags prevent missing values from being silently
# treated as genuine zeros. Imputation is fitted on training data only.
missing_flag_features = []
for c in numeric_features:
    flag = f"has_value_{c}"
    df[flag] = df[c].notna().astype(int)
    missing_flag_features.append(flag)

feature_columns = numeric_features + categorical_features + missing_flag_features
X = df[feature_columns].copy()
y = df["is_declining_label"].astype(int)
groups = df["client_id"].fillna("unknown").astype(str)

if y.nunique() != 2:
    raise ValueError("The target must contain both classes.")
if groups.nunique() < 5:
    raise ValueError("Not enough distinct clients for a useful group holdout.")

# Try deterministic client-grouped splits until both classes are represented
# in train and test. No client should occur on both sides.
split_found = False
for seed in range(RANDOM_STATE, RANDOM_STATE + 100):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups))
    if y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2:
        split_found = True
        break

if not split_found:
    raise ValueError(
        "Could not find a client-grouped split containing both target classes. "
        "Do not silently fall back to a row split."
    )

assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

print("Dataset shape:", df.shape)
print("Unique clients:", groups.nunique())
print("Positive-label rate:", round(y.mean(), 4))
print("Training rows:", len(train_idx), "| Test rows:", len(test_idx))
print("Train clients:", groups.iloc[train_idx].nunique(),
      "| Test clients:", groups.iloc[test_idx].nunique())
print("Train/test client overlap:", len(
    set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
))
print("Excluded leakage columns: trend_pct, trend_direction, is_declining_label")



Dataset shape: (30000, 67)
Unique clients: 32
Positive-label rate: 0.5421
Training rows: 23837 | Test rows: 6163
Train clients: 25 | Test clients: 7
Train/test client overlap: 0
Excluded leakage columns: trend_pct, trend_direction, is_declining_label


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a deterministic 80/20 client-grouped holdout with a fixed random seed. Entire clients are assigned to either training or test data, preventing the same client's records from appearing on both sides. Imputation and categorical encoding are fitted within the training pipeline. The test set is reserved for comparing the frozen Week 4 baseline and the learned models. This evaluates generalization to held-out clients in this dataset, not necessarily to future months or every client population.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 2 — Preprocessing fitted only on the training data

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features + missing_flag_features),
    ("categorical", categorical_transformer, categorical_features)
])

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Preprocessing is configured.")
print("Numeric features, including missingness flags:",
      len(numeric_features + missing_flag_features))
print("Categorical features:", len(categorical_features))



Preprocessing is configured.
Numeric features, including missingness flags: 36
Categorical features: 8


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare Logistic Regression and Random Forest using Precision@50 as the primary ranking metric. I also report Precision@20, Precision@100, average precision, and ROC AUC to show whether conclusions depend on the cutoff. The Week 4 baseline is reconstructed from the documented scoring formula using the same dataset, then evaluated on the same held-out test rows. The model is not considered better unless the measured comparison supports that conclusion.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 3 — Train models and compare with the Week 4 baseline

def precision_at_k(y_true, scores, k):
    y_arr = np.asarray(y_true)
    scores_arr = np.asarray(scores)
    n = min(k, len(y_arr))
    top_idx = np.argsort(-scores_arr, kind="stable")[:n]
    return float(y_arr[top_idx].mean()) if n else float("nan")

def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(pct=True)

def normalize(series):
    s = pd.to_numeric(series, errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

# Reconstruct the repository's documented Week 4 score.
baseline = df.copy()
baseline["visibility_score"] = percentile_rank(
    np.log1p(pd.to_numeric(baseline["impressions_90d"], errors="coerce").clip(lower=0))
)
baseline["freshness_risk_score"] = percentile_rank(
    baseline["days_since_last_update"]
)
baseline["position_opportunity_score"] = (
    (1 - normalize(pd.to_numeric(baseline["avg_position"], errors="coerce").clip(1, 50)))
    * baseline["visibility_score"]
    * (pd.to_numeric(baseline["avg_position"], errors="coerce") > 0).astype(int)
)
baseline["depth_gap_score"] = (
    1 - percentile_rank(baseline["word_count"])
) * baseline["visibility_score"]

baseline["baseline_refresh_score"] = (
    0.40 * baseline["visibility_score"]
    + 0.30 * baseline["freshness_risk_score"]
    + 0.25 * baseline["position_opportunity_score"]
    + 0.05 * baseline["depth_gap_score"]
).clip(0, 1)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

results = []
fitted_models = {}
test_scores = {}

def add_metrics(name, scores):
    scores = np.asarray(scores)
    row = {
        "Approach": name,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Precision@100": precision_at_k(y_test, scores, 100),
        "Average precision": average_precision_score(y_test, scores),
        "ROC AUC": roc_auc_score(y_test, scores)
    }
    results.append(row)

baseline_scores = baseline.iloc[test_idx]["baseline_refresh_score"].to_numpy()
test_scores["Week 4 baseline"] = baseline_scores
add_metrics("Week 4 baseline", baseline_scores)

for name, classifier in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    pipeline.fit(X_train, y_train)
    scores = pipeline.predict_proba(X_test)[:, 1]
    fitted_models[name] = pipeline
    test_scores[name] = scores
    add_metrics(name, scores)

results_df = pd.DataFrame(results).sort_values(
    "Precision@50", ascending=False
).reset_index(drop=True)

display(results_df.style.format({
    "Precision@20": "{:.3f}",
    "Precision@50": "{:.3f}",
    "Precision@100": "{:.3f}",
    "Average precision": "{:.3f}",
    "ROC AUC": "{:.3f}"
}))

baseline_p50 = results_df.loc[
    results_df["Approach"] == "Week 4 baseline", "Precision@50"
].iloc[0]
best_model_row = results_df[results_df["Approach"] != "Week 4 baseline"].iloc[0]

print("Primary metric: Precision@50")
print("Baseline Precision@50:", round(baseline_p50, 3))
print("Best model by Precision@50:", best_model_row["Approach"])
print("Best model Precision@50:", round(best_model_row["Precision@50"], 3))
print("Difference vs baseline:",
      round(best_model_row["Precision@50"] - baseline_p50, 3))


,Approach,Precision@20,Precision@50,Precision@100,Average precision,ROC AUC
0,Logistic Regression,0.700,0.720,0.710,0.605,0.615
1,Random Forest,0.550,0.560,0.560,0.592,0.611
2,Week 4 baseline,0.400,0.320,0.310,0.482,0.498


Primary metric: Precision@50
Baseline Precision@50: 0.32
Best model by Precision@50: Logistic Regression
Best model Precision@50: 0.72
Difference vs baseline: 0.4


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I inspect false positives and false negatives at the default 0.5 probability threshold, and compare the three highest-ranked features where the estimator exposes importance. A false positive is a page ranked as declining that does not have the observed label; a false negative is a declining page classified below the threshold. These cases may reflect noisy trends, differences between clients, or limitations of the available features. Feature importance indicates association used by this model, not causation.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Section 4 — Error analysis and feature importance

best_name = best_model_row["Approach"]
best_model = fitted_models[best_name]
best_probabilities = test_scores[best_name]
predicted = (best_probabilities >= 0.5).astype(int)

errors = df.iloc[test_idx][
    ["content_id", "client_id", "is_declining_label"]
].copy()
errors["predicted_probability"] = best_probabilities
errors["predicted_label"] = predicted
errors["error_type"] = np.select(
    [
        (errors["is_declining_label"].to_numpy() == 0) & (predicted == 1),
        (errors["is_declining_label"].to_numpy() == 1) & (predicted == 0)
    ],
    ["False positive", "False negative"],
    default="Correct"
)

print("Error counts:")
display(errors["error_type"].value_counts().rename_axis("Type").to_frame("Count"))

print("Three incorrect cases (IDs are pseudonymous):")
display(
    errors[errors["error_type"] != "Correct"]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
)

classifier = best_model.named_steps["classifier"]
transformed_feature_names = (
    best_model.named_steps["preprocessor"].get_feature_names_out()
)

if hasattr(classifier, "feature_importances_"):
    importance = classifier.feature_importances_
elif hasattr(classifier, "coef_"):
    importance = np.abs(classifier.coef_[0])
else:
    importance = np.zeros(len(transformed_feature_names))

importance_df = pd.DataFrame({
    "Feature": transformed_feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False)

print("Top 10 model features:")
display(importance_df.head(10))

print(
    "Interpretation: compare the measured ranking metrics before claiming "
    "the model improves on the baseline. These results describe historical "
    "declining-trend labels, not the causal effect of refreshing content."
)


Error counts:


,Count
Type,
Correct,3588
False positive,1416
False negative,1159


Three incorrect cases (IDs are pseudonymous):


,content_id,client_id,is_declining_label,predicted_probability,predicted_label,error_type
26614,content_7be5f150dc65,client_f369cb89fc,0,0.946053,1,False positive
20736,content_41baf0722ad9,client_8527a891e2,0,0.932678,1,False positive
12869,content_5d5653c4eb4f,client_4e07408562,0,0.917315,1,False positive


Top 10 model features:


,Feature,Importance
14,numeric__log_impressions_90d,1.424989
66,categorical__position_tier_top_3,0.782870
60,categorical__impression_tier_low,0.749944
58,categorical__impression_tier_excellent,0.597324
15,numeric__log_clicks_90d,0.570576
54,categorical__word_count_tier_1000-2000,0.549907
39,categorical__content_type_comparison article,0.548517
52,categorical__freshness_tier_31-90,0.480756
10,numeric__avg_position,0.386719
55,categorical__word_count_tier_2000-3500,0.364619


Interpretation: compare the measured ranking metrics before claiming the model improves on the baseline. These results describe historical declining-trend labels, not the causal effect of refreshing content.


### Results and interpretation

Logistic Regression achieved the highest reported Precision@50 of 0.720, compared with 0.560 for Random Forest and 0.320 for the reconstructed Week 4 baseline. This is an absolute improvement of 0.400 over the baseline on the evaluated client-grouped test split. Logistic Regression also achieved the highest reported Precision@20, Precision@100, average precision, and ROC AUC among the three approaches.

The Logistic Regression error analysis identified 1,416 false positives and 1,159 false negatives at the default 0.5 probability threshold, alongside 3,588 correct classifications. Three inspected false positives had predicted probabilities above 0.91, showing that high model confidence does not guarantee a correct prediction.

The largest reported absolute coefficient magnitudes were associated with log impressions, the top-three position category, impression tiers, and log clicks. These results suggest that visibility and traffic-related features contribute strongly to the model's predictions. They do not establish that these factors cause content to decline.

The ROC AUC of 0.615 indicates that overall discrimination remains limited, even though Precision@50 is comparatively strong. The model may be useful as a candidate for prioritizing content for further review, but its results should not be interpreted as proof that refreshing a page will improve future performance.

The Week 4 baseline was reconstructed from the documented scoring formula, so the comparison depends on that reconstruction matching the original implementation. Also, because the learned models were compared and selected using the held-out test results, the reported best-model performance may be optimistic. A stronger follow-up would select the model on validation data and reserve a separate test set for final evaluation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.